In [47]:
# ============================================================
# CELL 1 : OPEN PORTAL + SEARCH
# ============================================================

import time

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import Select
from webdriver_manager.chrome import ChromeDriverManager

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

URL = "https://gr.maharashtra.gov.in/1145/Government-Resolutions"

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install())
)

driver.maximize_window()

print("Opening Portal...")

driver.get(URL)

time.sleep(5)

# ------------------------------------------------------------
# SWITCH TO ENGLISH
# ------------------------------------------------------------

try:

    driver.find_element(
        By.ID,
        "HeaderMain1_SetCulture1_btn_Language"
    ).click()

    print("Switching to English...")

    time.sleep(5)

except:

    print("Already in English or language button not found.")

# ------------------------------------------------------------
# DEPARTMENT LIST
# ------------------------------------------------------------

dropdown = Select(
    driver.find_element(
        By.ID,
        "SitePH_ddlDepartmentType"
    )
)

print("\nAvailable Departments\n")
print("-" * 70)

for op in dropdown.options:
    print(op.text)

print("-" * 70)

department = input("\nEnter Department Name exactly as shown: ").strip()

dropdown.select_by_visible_text(department)

print("Selected:", department)

# ------------------------------------------------------------
# DATE RANGE
# ------------------------------------------------------------

driver.execute_script("""
var f=document.getElementById('SitePH_txtFromDate');
var t=document.getElementById('SitePH_txtToDate');

f.removeAttribute('readonly');
t.removeAttribute('readonly');

f.value='01/01/2020';
t.value='31/12/2024';

f.dispatchEvent(new Event('change',{bubbles:true}));
t.dispatchEvent(new Event('change',{bubbles:true}));
""")

time.sleep(1)

print("From Date :", driver.find_element(By.ID,"SitePH_txtFromDate").get_attribute("value"))
print("To Date   :", driver.find_element(By.ID,"SitePH_txtToDate").get_attribute("value"))

print("\n===========================================")
print("NOW:")
print("1. Solve the CAPTCHA")
print("2. Click SEARCH yourself")
print("3. Wait until the results table loads")
print("4. Come back here")
print("===========================================")

input("\nPress ENTER only AFTER results are visible...")

# ------------------------------------------------------------
# VERIFY RESULTS
# ------------------------------------------------------------

table = driver.find_element(By.ID, "SitePH_dgvDocuments")

rows = table.find_elements(By.TAG_NAME, "tr")

print(f"\nResults Loaded Successfully.")
print(f"Rows on current page : {len(rows)-1}")

print("\nCell 1 Complete.")

Opening Portal...
Switching to English...

Available Departments

----------------------------------------------------------------------
-- Select --
Agriculture Department
Agriculture, Dairy Development, Animal Husbandry and Fisheries Department
Animal Husbandry, Dairy Development & Fisheries Department
Co-operation Department
Co-operation, Textiles and Marketing Department
Cultural Affairs Department
Electronics, Information Technology and Artificial Intelligence Department
Energy Department
Environment Department
Finance Department
Food and Drug Administration Department
Food, Civil Supplies and Consumer Protection Department
Forest Department
General Administration Department
Higher and Technical Education Department
Home Department
Housing Department
Industries and Mining Department
Industries, Energy and Labour Department
Information and Public Relations Department
Labour Department
Law and Judiciary Department
Marathi Language Department
Marketing Department
Medical Education an


Enter Department Name exactly as shown:  Minorities Development Department


Selected: Minorities Development Department
From Date : 01/01/2020
To Date   : 31/12/2024

NOW:
1. Solve the CAPTCHA
2. Click SEARCH yourself
3. Wait until the results table loads
4. Come back here



Press ENTER only AFTER results are visible... 



Results Loaded Successfully.
Rows on current page : 10

Cell 1 Complete.


In [49]:
# ============================================================
# CELL 2 : EXTRACT ALL METADATA FROM ALL PAGES
# ============================================================

import os
import time
import pandas as pd

from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException

metadata = []

page = 1

while True:

    print(f"\n========== PAGE {page} ==========")

    time.sleep(2)

    table = driver.find_element(By.ID, "SitePH_dgvDocuments")

    rows = table.find_elements(By.TAG_NAME, "tr")

    print("Rows :", len(rows)-1)

    # Skip header row
    for row in rows[1:]:

        cols = row.find_elements(By.TAG_NAME, "td")

        if len(cols) < 7:
            continue

        department = cols[1].text.strip()
        title = cols[2].text.strip()
        unique_code = cols[3].text.strip()
        gr_date = cols[4].text.strip()
        pdf_size = cols[5].text.strip()

        pdf_link = ""

        try:

            a = cols[6].find_element(By.TAG_NAME, "a")

            pdf_link = a.get_attribute("href")

        except:
            pass

        metadata.append({

            "Department": department,
            "Title": title,
            "Unique_Code": unique_code,
            "GR_Date": gr_date,
            "Website_Size": pdf_size,
            "PDF_URL": pdf_link

        })

    print("Collected :", len(metadata))

    # =======================================================
    # NEXT PAGE
    # =======================================================

    try:

        next_button = driver.find_element(
            By.ID,
            "SitePH_ucPaging_lnkNext"
        )

        if not next_button.is_enabled():
            break

        driver.execute_script(
            "arguments[0].scrollIntoView();",
            next_button
        )

        time.sleep(1)

        next_button.click()

        page += 1

        time.sleep(3)

    except NoSuchElementException:

        print("Last Page Reached.")

        break

# ============================================================
# SAVE MASTER METADATA
# ============================================================

df = pd.DataFrame(metadata)

df.to_csv(
    "master_metadata.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n===================================")
print("Finished Metadata Extraction")
print("Total GRs :", len(df))
print("Saved : master_metadata.csv")
print("===================================")

display(df.head())


========== PAGE 1 ==========
Rows : 10
Collected : 10

========== PAGE 2 ==========
Rows : 10
Collected : 20

========== PAGE 3 ==========
Rows : 10
Collected : 30

========== PAGE 4 ==========
Rows : 10
Collected : 40

========== PAGE 5 ==========
Rows : 10
Collected : 50

========== PAGE 6 ==========
Rows : 10
Collected : 60

========== PAGE 7 ==========
Rows : 10
Collected : 70

========== PAGE 8 ==========
Rows : 10
Collected : 80

========== PAGE 9 ==========
Rows : 10
Collected : 90

========== PAGE 10 ==========
Rows : 10
Collected : 100

========== PAGE 11 ==========
Rows : 10
Collected : 110

========== PAGE 12 ==========
Rows : 10
Collected : 120

========== PAGE 13 ==========
Rows : 10
Collected : 130

========== PAGE 14 ==========
Rows : 10
Collected : 140

========== PAGE 15 ==========
Rows : 10
Collected : 150

========== PAGE 16 ==========
Rows : 10
Collected : 160

========== PAGE 17 ==========
Rows : 10
Collected : 170

========== PAGE 18 ==========
Rows : 10
Collecte

,Department,Title,Unique_Code,GR_Date,Website_Size,PDF_URL
0,Minorities Development Department,Regarding distribution of Grant to the Maharas...,202412311514236414,31-12-2024,342,https://gr.maharashtra.gov.in/Site/Upload/Gove...
1,Minorities Development Department,Regarding providing administrative approval fo...,202412271752201814,27-12-2024,167,https://gr.maharashtra.gov.in/Site/Upload/Gove...
2,Minorities Development Department,Regarding declaration of Under Secretary (Desk...,202412171445512714,17-12-2024,239,https://gr.maharashtra.gov.in/Site/Upload/Gove...
3,Minorities Development Department,Regarding celebrating December 18 as Minority ...,202412161226382314,16-12-2024,216,https://gr.maharashtra.gov.in/Site/Upload/Gove...
4,Minorities Development Department,Scholarship Scheme for students from minority ...,202412131525452814,13-12-2024,319,https://gr.maharashtra.gov.in/Site/Upload/Gove...


In [51]:
# ============================================================
# CELL 3 : DOWNLOAD ALL PDFs
# ============================================================

import os
import time
import fitz
import requests
import pandas as pd

# ============================================================
# CONFIG
# ============================================================

ROOT_FOLDER = "GR_DATASET"

PDF_FOLDER = os.path.join(ROOT_FOLDER, "pdfs")
META_FOLDER = os.path.join(ROOT_FOLDER, "metadata")

os.makedirs(PDF_FOLDER, exist_ok=True)
os.makedirs(META_FOLDER, exist_ok=True)

# ============================================================
# LOAD METADATA
# ============================================================

df = pd.read_csv("master_metadata.csv")

download_status = []
pages_list = []
actual_size = []
local_files = []
download_time = []

print(f"Downloading {len(df)} PDFs...\n")

# ============================================================
# DOWNLOAD LOOP
# ============================================================

for idx, row in df.iterrows():

    dept = str(row["Department"]).strip()

    # Safe folder name
    dept = dept.replace("/", "-").replace("\\", "-").replace(":", "-")

    dept_folder = os.path.join(PDF_FOLDER, dept)

    os.makedirs(dept_folder, exist_ok=True)

    pdf_url = row["PDF_URL"]

    unique = str(row["Unique_Code"]).strip()

    filename = os.path.join(
        dept_folder,
        unique + ".pdf"
    )

    print(f"[{idx+1}/{len(df)}] {unique}")

    try:

        response = requests.get(
            pdf_url,
            timeout=60
        )

        response.raise_for_status()

        with open(filename, "wb") as f:
            f.write(response.content)

        # PDF Information
        doc = fitz.open(filename)

        pages = len(doc)

        doc.close()

        size = os.path.getsize(filename)

        status = "Downloaded"

    except Exception as e:

        print("FAILED :", e)

        pages = 0
        size = 0
        status = str(e)

    download_status.append(status)
    pages_list.append(pages)
    actual_size.append(size)
    local_files.append(filename)
    download_time.append(
        time.strftime("%Y-%m-%d %H:%M:%S")
    )

# ============================================================
# UPDATE DATAFRAME
# ============================================================

df["Pages"] = pages_list
df["Downloaded_Size(Bytes)"] = actual_size
df["Local_File"] = local_files
df["Download_Status"] = download_status
df["Downloaded_On"] = download_time

# ============================================================
# SAVE MASTER CSV
# ============================================================

master_csv = os.path.join(
    ROOT_FOLDER,
    "master_metadata.csv"
)

df.to_csv(
    master_csv,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# SAVE DEPARTMENT CSV
# ============================================================

for dept, group in df.groupby("Department"):

    safe = dept.replace("/", "-").replace("\\", "-").replace(":", "-")

    group.to_csv(
        os.path.join(
            META_FOLDER,
            safe + ".csv"
        ),
        index=False,
        encoding="utf-8-sig"
    )

# ============================================================
# SUMMARY
# ============================================================

success = (df["Download_Status"] == "Downloaded").sum()
failed = len(df) - success

print("\n==========================================")
print("DOWNLOAD COMPLETE")
print("==========================================")
print("Total PDFs      :", len(df))
print("Downloaded      :", success)
print("Failed          :", failed)
print("Dataset Folder  :", ROOT_FOLDER)
print("==========================================")


[1/784] 202412311514236414
[2/784] 202412271752201814
[3/784] 202412171445512714
[4/784] 202412161226382314
[5/784] 202412131525452814
[6/784] 202411291137374014
[7/784] 202411281525130314
[8/784] 202411271103546714
[9/784] 202411271425446814
[10/784] 202411271716198714...
[11/784] 202411251624051514
[12/784] 202410141649348214
[13/784] 202410111829139114
[14/784] 202410111936170614....
[15/784] 202410101722119714....
[16/784] 202410101726377414
[17/784] 202410101730038214
[18/784] 202410101751397114
[19/784] 202410091406399514
[20/784] 202410071721546214
[21/784] 202410071612471814
[22/784] 202410071701458114
[23/784] 202410071705037514
[24/784] 202410071707023914
[25/784] 202410041251451714
[26/784] 202410041512268214....
[27/784] 202410041512567514
[28/784] 202410042106343614...
[29/784] 202410011852022414
[30/784] 202409271437583214..
[31/784] 202409241641523614
[32/784] 202409231457234514
[33/784] 202409231503165714
[34/784] 202409201518345014
[35/784] 202409201742505614
[36/784]